# Simple LangChain RAG with the same PDFThis notebook uses **`metagpt.pdf`** in the same folder as the notebook.It avoids legacy `langchain.chains` imports and uses a plain retrieval + prompt flow.

In [ ]:
# Cell 1: install packages!pip install -qU langchain langchain-openai langchain-community langchain-text-splitters pypdf python-dotenv

In [ ]:
# Cell 2: load API keyimport osfrom dotenv import load_dotenvload_dotenv()OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")if not OPENAI_API_KEY:    raise ValueError("OPENAI_API_KEY not found. Put it in your .env file or environment.")

In [ ]:
# Cell 3: importsfrom langchain_openai import ChatOpenAI, OpenAIEmbeddingsfrom langchain_community.document_loaders import PyPDFLoaderfrom langchain_text_splitters import RecursiveCharacterTextSplitterfrom langchain_core.vectorstores import InMemoryVectorStore

In [ ]:
# Cell 4: load the same PDFloader = PyPDFLoader("metagpt.pdf")docs = loader.load()print("Pages loaded:", len(docs))print(docs[0].metadata if docs else "No pages found")

In [ ]:
# Cell 5: split the PDF into chunkssplitter = RecursiveCharacterTextSplitter(    chunk_size=1000,    chunk_overlap=200)chunks = splitter.split_documents(docs)print("Chunks created:", len(chunks))print(chunks[0].page_content[:500] if chunks else "No chunks found")

In [ ]:
# Cell 6: create embeddings, vector store, and retrieverembeddings = OpenAIEmbeddings(    model="text-embedding-3-small",    api_key=OPENAI_API_KEY)vectorstore = InMemoryVectorStore.from_documents(    documents=chunks,    embedding=embeddings)retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

In [ ]:
# Cell 7: create the chat modelllm = ChatOpenAI(    model="gpt-5.4-nano",    api_key=OPENAI_API_KEY)

In [ ]:
# Cell 8: simple RAG function (no langchain.chains, no ChatPromptTemplate)def ask_pdf(question: str):    retrieved_docs = retriever.invoke(question)    context = "\n\n".join([doc.page_content for doc in retrieved_docs])    prompt = f"""Answer the question using only the context below.If the answer is not in the context, say: I do not know based on the PDF.Context:{context}Question:{question}"""    response = llm.invoke(prompt)    print("ANSWER:\n")    print(response.content)    print("\n" + "=" * 80)    print("TOP SOURCES:\n")    for i, doc in enumerate(retrieved_docs, 1):        page = doc.metadata.get("page", "unknown")        source = doc.metadata.get("source", "unknown")        print(f"[{i}] page={page} | source={source}")        print(doc.page_content[:300])        print("-" * 80)    return response, retrieved_docs

In [ ]:
# Cell 9: test questionresponse, retrieved_docs = ask_pdf("What is the summary of this paper?")

In [ ]:
# Cell 10: another questionresponse, retrieved_docs = ask_pdf("How do agents share information with other agents?")

In [ ]:
# Cell 11: another questionresponse, retrieved_docs = ask_pdf("Tell me about the ablation study results.")